In [2]:
# Intelligent Cyber Threat Detection using Autoencoder
# Dataset: CICIDS2017 (Monday)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from soc_engine.scorer import SeverityScorer
from soc_engine.temporal_engine import apply_temporal_smoothing

In [5]:
import numpy as np

errors = np.load("errors.npy")

print("Loaded errors:", len(errors))

Loaded errors: 21176


In [6]:
print("Min error:", np.min(errors))
print("Max error:", np.max(errors))
print("Mean error:", np.mean(errors))

Min error: 0.012290916
Max error: 402.75403
Mean error: 0.15156104


In [8]:
# split data
split_idx = int(0.7 * len(errors))

train_errors = errors[:split_idx]   # assume mostly normal
test_errors = errors[split_idx:]

# fit only on train
scorer = SeverityScorer()
scorer.fit(train_errors)

# score only on test
results = scorer.score(test_errors)

print("Scoring complete. Sample:\n")

for r in results[:5]:
    print(r)

Scoring complete. Sample:

{'window_index': 0, 'severity_score': -0.587968, 'severity_label': 'NORMAL', 'is_anomaly': 0, 'confidence': 0.2152}
{'window_index': 1, 'severity_score': -0.403805, 'severity_label': 'NORMAL', 'is_anomaly': 0, 'confidence': 0.2311}
{'window_index': 2, 'severity_score': -0.331279, 'severity_label': 'NORMAL', 'is_anomaly': 0, 'confidence': 0.2376}
{'window_index': 3, 'severity_score': 0.522188, 'severity_label': 'NORMAL', 'is_anomaly': 0, 'confidence': 0.3232}
{'window_index': 4, 'severity_score': 1.504678, 'severity_label': 'NORMAL', 'is_anomaly': 0, 'confidence': 0.4384}


In [9]:
from collections import Counter

labels = [r["severity_label"] for r in results]

print("Label Distribution:")
print(Counter(labels))

Label Distribution:
Counter({'NORMAL': 6092, 'LOW': 125, 'HIGH': 68, 'MEDIUM': 68})


In [10]:
print("First few non-normal samples:\n")

for r in results:
    if r["severity_label"] != "NORMAL":
        print(r)
        break

First few non-normal samples:

{'window_index': 33, 'severity_score': 19.351844, 'severity_label': 'HIGH', 'is_anomaly': 1, 'confidence': 0.9998}


In [11]:
from soc_engine.temporal_engine import apply_temporal_smoothing

smoothed_results = apply_temporal_smoothing(results)

In [13]:
print("First 10 non-normal samples:\n")

count = 0
for r in results:
    if r["severity_label"] != "NORMAL":
        print(r)
        count += 1
    if count == 10:
        break

First 10 non-normal samples:

{'window_index': 33, 'severity_score': 19.351844, 'severity_label': 'HIGH', 'is_anomaly': 1, 'confidence': 0.9998}
{'window_index': 82, 'severity_score': 19.427912, 'severity_label': 'HIGH', 'is_anomaly': 1, 'confidence': 0.9998}
{'window_index': 120, 'severity_score': 5.008598, 'severity_label': 'MEDIUM', 'is_anomaly': 1, 'confidence': 0.8182}
{'window_index': 121, 'severity_score': 4.427687, 'severity_label': 'MEDIUM', 'is_anomaly': 1, 'confidence': 0.771}
{'window_index': 122, 'severity_score': 4.268024, 'severity_label': 'MEDIUM', 'is_anomaly': 1, 'confidence': 0.7566}
{'window_index': 123, 'severity_score': 3.943624, 'severity_label': 'LOW', 'is_anomaly': 1, 'confidence': 0.7255}
{'window_index': 183, 'severity_score': 24.148964, 'severity_label': 'HIGH', 'is_anomaly': 1, 'confidence': 1.0}
{'window_index': 184, 'severity_score': 4.851357, 'severity_label': 'MEDIUM', 'is_anomaly': 1, 'confidence': 0.8062}
{'window_index': 185, 'severity_score': 4.9866

In [14]:
print("Temporal behavior around anomalies:\n")

for i in range(115, 130):
    print(
        i,
        results[i]["severity_label"],
        "→",
        smoothed_results[i]["temporal_label"]
    )

Temporal behavior around anomalies:

115 NORMAL → NORMAL
116 NORMAL → NORMAL
117 NORMAL → NORMAL
118 NORMAL → NORMAL
119 NORMAL → NORMAL
120 MEDIUM → NORMAL
121 MEDIUM → MEDIUM
122 MEDIUM → MEDIUM
123 LOW → MEDIUM
124 NORMAL → MEDIUM
125 NORMAL → MEDIUM
126 NORMAL → NORMAL
127 NORMAL → NORMAL
128 NORMAL → NORMAL
129 NORMAL → NORMAL


In [15]:
from soc_engine.temporal_engine import apply_temporal_smoothing

slow_attack_sequence = [
    {"severity_score": 2.2, "severity_label": "LOW"},
    {"severity_score": 2.5, "severity_label": "LOW"},
    {"severity_score": 2.8, "severity_label": "LOW"},
    {"severity_score": 3.1, "severity_label": "LOW"},
    {"severity_score": 2.9, "severity_label": "LOW"},
    {"severity_score": 2.7, "severity_label": "LOW"},
]

smoothed_slow_attack = apply_temporal_smoothing(slow_attack_sequence)

print("Slow Attack Simulation:\n")

for i, r in enumerate(smoothed_slow_attack):
    print(f"Step {i}: {r['temporal_label']}")
    

Slow Attack Simulation:

Step 0: NORMAL
Step 1: NORMAL
Step 2: LOW
Step 3: LOW
Step 4: LOW
Step 5: LOW


In [16]:
print("Without temporal logic:")
for s in slow_attack_sequence:
    print(s["severity_label"])

Without temporal logic:
LOW
LOW
LOW
LOW
LOW
LOW
